In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd

import folium

In [2]:
df = pd.read_parquet("branch.parquet", engine="fastparquet")

In [3]:
df.head()

,BranchID,BranchCode,BranchType,IsActive,BranchName,AMBadgeNo,AreaManager,Region,RMBadgeNo,RegionalManager,Longitude,Latitude,City,StationBrand
0,4BC3ADA4-5793-4026-B2A6-2C14ABAD2305,1884,PE,False,TRISTAR DAMMAM,None,None,PAC,None,None,NaN,NaN,DAMMAM,PE
1,92DB0871-73E5-423A-AE8B-2E49A2A52058,1583,PE,False,ALDRESS MAROOJ,None,None,CR NORTH,None,None,NaN,NaN,RIYADH,PE
2,F5DD72D1-D6EA-4C45-989C-3499BF156D25,1508,PE,False,TASHHEED AL RIYADH,None,None,CR EAST,None,None,NaN,NaN,RIYADH,PE
3,429D8BE2-F6DC-45D4-8B52-43CF2E6B654D,1119,PE,False,ZTAMWAJ,None,None,SOUTHERN REGION,None,None,NaN,NaN,ABHA,PE
4,0B5F67FA-1D34-42E1-825D-454AF1365037,2089_ToDex,Diesel,False,Well Qassim-Riyadh-Al Atsh,None,None,None,None,None,25.525793,45.946956,None,PE


In [4]:
df["Latitude"].min(), df["Latitude"].max()

(0.0, 50.220000000000006)

In [5]:
df["Longitude"].min(), df["Longitude"].max()

(0.0, 50.216111000000005)

In [6]:
df["new_latitude"] = np.where(df["Latitude"] > df["Longitude"], df["Longitude"], df["Latitude"])
df["Longitude"] = np.where(df["Latitude"] > df["Longitude"], df["Latitude"], df["Longitude"])
df["Latitude"] = df["new_latitude"].copy()
df = df.drop(columns="new_latitude")

In [7]:
cols = df.columns.values
cols = list(cols)
cols.remove("StationBrand")

In [8]:
cols

['BranchID',
 'BranchCode',
 'BranchType',
 'IsActive',
 'BranchName',
 'AMBadgeNo',
 'AreaManager',
 'Region',
 'RMBadgeNo',
 'RegionalManager',
 'Longitude',
 'Latitude',
 'City']

In [9]:
df.drop_duplicates(subset=cols)["Region"].value_counts()

Region
WR SOUTH           141
WR NORTH           141
CR SOUTH           137
CR EAST            124
CR NORTH           115
SOUTHERN REGION     99
ER NORTH            96
ER SOUTH            90
PAC                 24
DIESEL               9
Name: count, dtype: int64

In [10]:
df.drop_duplicates(subset=cols).shape, df.shape

((997, 14), (997, 14))

In [11]:
df.drop_duplicates(subset=cols).to_parquet("../data/01_raw/branches.parquet", engine="fastparquet", index=False)
df.drop_duplicates(subset=cols).to_parquet("branch.parquet", engine="fastparquet", index=False)

In [12]:
df.isna().mean()

BranchID           0.000000
BranchCode         0.000000
BranchType         0.110331
IsActive           0.000000
BranchName         0.000000
AMBadgeNo          0.098295
AreaManager        0.098295
Region             0.021063
RMBadgeNo          0.162487
RegionalManager    0.162487
Longitude          0.153460
Latitude           0.153460
City               0.024072
StationBrand       0.000000
dtype: float64

In [257]:
import pickle
import joblib

In [258]:
import sys

In [259]:
sys.path.append('../src/')

In [260]:
pipe = joblib.load("../data/06_models/segmentation/transactions/segmentation_model_artifact.pkl")

In [261]:
train_df = pd.read_parquet("../data/05_model_input/segmentation/transactions/segmentation_master_filtered", engine="pyarrow")

In [262]:
cols = ['most_trx_branch_n_places_within_1000m__All', 'most_trx_branch_n_places_within_1000m__community', 'avg_n_places_within_1000m__All', 'trx_avg_skus_per_order_mean_past_2_next_0_months', 'trx_total_profit_mean_past_2_next_0_months', 'avg_n_places_within_1000m__community', 'avg_pop_density', 'trx_total_mileage_sum_past_2_next_0_months', 'trx_avg_delta_mileage_mean_past_2_next_0_months', 'most_trx_branch_pop_density']

In [263]:
train_df[cols].head()

,most_trx_branch_n_places_within_1000m__All,most_trx_branch_n_places_within_1000m__community,avg_n_places_within_1000m__All,trx_avg_skus_per_order_mean_past_2_next_0_months,trx_total_profit_mean_past_2_next_0_months,avg_n_places_within_1000m__community,avg_pop_density,trx_total_mileage_sum_past_2_next_0_months,trx_avg_delta_mileage_mean_past_2_next_0_months,most_trx_branch_pop_density
0,7.666667,3.0,7.666667,1.000000,121.993695,3.0,2189.769287,26550.0,2950.000000,2189.769287
1,0.000000,0.0,0.000000,1.333333,32.146633,0.0,319.372650,100424.0,8368.666667,319.372650
2,0.000000,0.0,0.000000,1.000000,29.793332,0.0,2548.355469,420204.0,46689.333333,2548.355469
3,0.000000,0.0,0.000000,2.000000,140.935431,0.0,5.676630,174648.0,9702.666667,5.676630
4,0.000000,0.0,0.000000,1.000000,86.398034,0.0,290.096741,32172.0,3574.666667,290.096741


In [264]:
train_df.head()

,_id,_observ_end_dt,trx_total_sales,trx_net_sales,trx_total_profit,trx_total_cost,trx_total_amount_discounts,trx_total_qty_discounts,trx_total_mileage,trx_avg_delta_mileage,...,trx_distinct_product_categories_mean_past_2_next_0_months,trx_distinct_branches_mean_past_11_next_0_months,trx_distinct_branches_mean_past_5_next_0_months,trx_distinct_branches_mean_past_2_next_0_months,trx_distinct_skus_sold_mean_past_11_next_0_months,trx_distinct_skus_sold_mean_past_5_next_0_months,trx_distinct_skus_sold_mean_past_2_next_0_months,trx_distinct_transactions_mean_past_11_next_0_months,trx_distinct_transactions_mean_past_5_next_0_months,trx_distinct_transactions_mean_past_2_next_0_months
0,0001215A-C5A0-4FA1-AB6E-AA226BB49C85,2024-12-31,527.789999,458.950006,365.981085,84.278900,21.910000,1,26550.0,8850.0,...,1.000000,0.250000,0.166667,0.333333,0.583333,0.500000,1.000000,0.250000,0.166667,0.333333
1,00016036-55B8-410C-B290-C45F5ADBC6F6,2024-12-31,178.490002,155.210000,96.439899,49.990099,20.440001,1,100424.0,25106.0,...,1.000000,0.250000,0.166667,0.333333,0.666667,0.666667,1.333333,0.250000,0.166667,0.333333
2,0001CEC2-8318-4E0F-B83B-6FD9315D5159,2024-12-31,165.010000,143.480003,89.379997,34.099998,0.000000,0,420204.0,140068.0,...,0.666667,0.083333,0.166667,0.333333,0.250000,0.500000,1.000000,0.083333,0.166667,0.333333
3,0001F611-8A45-4A5D-9477-F3D5DCF417FD,2024-12-31,662.279987,575.890002,422.806292,144.393701,65.410000,2,174648.0,29108.0,...,1.666667,0.083333,0.166667,0.333333,0.500000,1.000000,2.000000,0.083333,0.166667,0.333333
4,00027EBF-F533-4ABA-9087-AD039503378A,2024-12-31,360.010000,313.049998,259.194103,45.155902,0.000000,0,32172.0,10724.0,...,1.000000,0.250000,0.333333,0.333333,1.000000,1.500000,1.000000,0.250000,0.333333,0.333333


In [265]:
train_df["cluster_id"] = pipe.predict(train_df)

In [266]:
branch_cluster_df = train_df.groupby(
    ["cluster_id", "most_trx_branch_branch_id"],
    as_index=False
).agg(
    customer_count=("_id", "count")
)

branch_cluster_df["total_customer"] = branch_cluster_df.groupby("most_trx_branch_branch_id")["customer_count"].transform("sum")
branch_cluster_df["max_customer"] = branch_cluster_df.groupby("most_trx_branch_branch_id")["customer_count"].transform("max")
branch_cluster_df["cluster_share"] = branch_cluster_df["customer_count"] / branch_cluster_df["total_customer"]

In [267]:
branch_cluster_pivot_df = branch_cluster_df.pivot_table(index="most_trx_branch_branch_id", columns="cluster_id", values="cluster_share", fill_value=0,)

In [268]:

branch_cluster_pivot_df.columns = [f"cluster_{col}_trx_share" for col in branch_cluster_pivot_df.columns]
branch_cluster_pivot_df.index.name = "BranchID"
branch_cluster_pivot_df = branch_cluster_pivot_df.reset_index()
branch_cluster_pivot_df.index.name = None

In [269]:
branch_cluster_pivot_df

,BranchID,cluster_0_trx_share,cluster_1_trx_share,cluster_2_trx_share,cluster_3_trx_share,cluster_4_trx_share,cluster_5_trx_share
0,006D8F31-428E-4FFB-98D4-9C3AE2F1B4EE,0.000000,0.000000,0.000000,0.083045,0.916955,0.000000
1,01D07F68-67A3-42F1-B2E5-694028D89272,0.000000,0.000000,0.817102,0.007126,0.000000,0.175772
2,01E30C30-B4B6-4087-B994-1509989401AD,0.000000,0.000000,0.879245,0.011321,0.000000,0.109434
3,01E52D08-5F0C-49EA-B05C-A9AE8CAF5F6B,0.000000,0.000000,0.683871,0.000000,0.000000,0.316129
4,01E7599A-E311-45B2-BA7D-15BF3AE149DC,0.000000,0.981132,0.000000,0.000000,0.000000,0.018868
...,...,...,...,...,...,...,...
620,FF0B0C86-1E9A-4671-953E-A75327302A36,0.927393,0.000000,0.000000,0.052805,0.000000,0.019802
621,FF0D1E8D-B3B8-4C13-80F8-6EFEFC42BD92,0.000000,0.000000,0.000000,0.036697,0.963303,0.000000
622,FF29F235-85D9-4FBC-B1C3-79B39D6D8DBB,0.000000,0.000000,0.000000,0.174194,0.825806,0.000000
623,FFBCD6BF-8787-4E57-945E-16A0A74963BC,0.000000,0.000000,0.000000,0.076923,0.923077,0.000000


In [270]:
filtered_branch_cluster_df = branch_cluster_df[branch_cluster_df["customer_count"] == branch_cluster_df["max_customer"]][["cluster_id", "most_trx_branch_branch_id"]].copy()
filtered_branch_cluster_df = filtered_branch_cluster_df.set_index("most_trx_branch_branch_id")
filtered_branch_cluster_df.index.name = "BranchID"
filtered_branch_cluster_df = filtered_branch_cluster_df.reset_index()

In [271]:
map_df = pd.merge(
    df,
    filtered_branch_cluster_df,
    on="BranchID",
    how="left"
)[["BranchID", "BranchCode", "BranchName", "City", "Latitude", "Longitude", "cluster_id"]]

map_df = pd.merge(
    map_df,
    branch_cluster_pivot_df,
    on="BranchID",
    how="left"
) 
map_df

,BranchID,BranchCode,BranchName,City,Latitude,Longitude,cluster_id,cluster_0_trx_share,cluster_1_trx_share,cluster_2_trx_share,cluster_3_trx_share,cluster_4_trx_share,cluster_5_trx_share
0,4BC3ADA4-5793-4026-B2A6-2C14ABAD2305,1884,TRISTAR DAMMAM,DAMMAM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,92DB0871-73E5-423A-AE8B-2E49A2A52058,1583,ALDRESS MAROOJ,RIYADH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,F5DD72D1-D6EA-4C45-989C-3499BF156D25,1508,TASHHEED AL RIYADH,RIYADH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,429D8BE2-F6DC-45D4-8B52-43CF2E6B654D,1119,ZTAMWAJ,ABHA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0B5F67FA-1D34-42E1-825D-454AF1365037,2089_ToDex,Well Qassim-Riyadh-Al Atsh,None,25.525793,45.946956,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
976,94CDEA02-B25D-4248-A9FF-D3EAFD40FF29,613,PAC Tahsilat Al Shaqeeq,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
977,A86C7C8C-7A3E-40E4-96EA-D57B1F6EE0C2,126,PAC Daham Land,JEDDAH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
978,A14ECB83-84EC-4E6C-99C4-E6891C3A0555,133,AL Jabria Centre,YANBU,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
979,6A00760C-AC4C-40FE-B38A-ED4DE83D0CEE,376,PAC Unaizah 2,UNAIZAH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [272]:
map_df.cluster_id.value_counts()

cluster_id
4.0    329
0.0    135
2.0    124
1.0     40
Name: count, dtype: int64

In [194]:
map_df.to_excel("../data/08_reporting/branch_cluster_shares.xlsx", engine="openpyxl")

In [273]:
geometry = gpd.points_from_xy(map_df["Longitude"], map_df["Latitude"], crs="EPSG:4326")
branch_gdf = gpd.GeoDataFrame(map_df, geometry=geometry,)


In [274]:
branch_gdf

,BranchID,BranchCode,BranchName,City,Latitude,Longitude,cluster_id,cluster_0_trx_share,cluster_1_trx_share,cluster_2_trx_share,cluster_3_trx_share,cluster_4_trx_share,cluster_5_trx_share,geometry
0,4BC3ADA4-5793-4026-B2A6-2C14ABAD2305,1884,TRISTAR DAMMAM,DAMMAM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT EMPTY
1,92DB0871-73E5-423A-AE8B-2E49A2A52058,1583,ALDRESS MAROOJ,RIYADH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT EMPTY
2,F5DD72D1-D6EA-4C45-989C-3499BF156D25,1508,TASHHEED AL RIYADH,RIYADH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT EMPTY
3,429D8BE2-F6DC-45D4-8B52-43CF2E6B654D,1119,ZTAMWAJ,ABHA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT EMPTY
4,0B5F67FA-1D34-42E1-825D-454AF1365037,2089_ToDex,Well Qassim-Riyadh-Al Atsh,None,25.525793,45.946956,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (45.94696 25.52579)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
976,94CDEA02-B25D-4248-A9FF-D3EAFD40FF29,613,PAC Tahsilat Al Shaqeeq,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT EMPTY
977,A86C7C8C-7A3E-40E4-96EA-D57B1F6EE0C2,126,PAC Daham Land,JEDDAH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT EMPTY
978,A14ECB83-84EC-4E6C-99C4-E6891C3A0555,133,AL Jabria Centre,YANBU,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT EMPTY
979,6A00760C-AC4C-40FE-B38A-ED4DE83D0CEE,376,PAC Unaizah 2,UNAIZAH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT EMPTY


In [136]:
import geopy

In [275]:
branch_gdf[branch_gdf["Latitude"] > 0]

,BranchID,BranchCode,BranchName,City,Latitude,Longitude,cluster_id,cluster_0_trx_share,cluster_1_trx_share,cluster_2_trx_share,cluster_3_trx_share,cluster_4_trx_share,cluster_5_trx_share,geometry
4,0B5F67FA-1D34-42E1-825D-454AF1365037,2089_ToDex,Well Qassim-Riyadh-Al Atsh,None,25.525793,45.946956,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (45.94696 25.52579)
19,AC88A9EF-5161-4D88-8101-6AA8629E7908,1760_ToDex,EXIT 18 DIESEL STATION,None,24.606428,46.832750,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (46.83275 24.60643)
32,7D61345C-8119-474C-965C-8723DECE3E7A,2039_ToDex,AL MUSTAQABAL DIESEL STATION,None,22.453806,39.191028,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (39.19103 22.45381)
33,A2B1D5E6-6713-4760-A950-8A7B0B41BBA1,1821_ToDex,AL HENDI DIESEL STATION,None,26.368911,50.027281,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (50.02728 26.36891)
35,6E84522C-BA66-4545-AC25-92D9C438174C,1213_ToDex,AL JAZEERA DIESEL STATION,None,22.203333,39.117222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (39.11722 22.20333)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
953,AD7EE15B-2F65-4679-BB04-B916BFE54D0B,1349,BSH-IND-FURSAN-SR-,BISHA,20.003611,42.614167,4.0,0.0,0.0,0.0,0.101190,0.898810,0.0,POINT (42.61417 20.00361)
954,87BE27E2-7CA6-48E4-8CF5-B92527E9607B,1243,ABH-NFT-HIZAM-SR-,ABHA,18.207778,42.491389,4.0,0.0,0.0,0.0,0.080645,0.919355,0.0,POINT (42.49139 18.20778)
955,62FCD4AA-53B1-4577-AFEF-9DB682E5E2C0,1899,BAI-STD-BAISH-SR-,BAISH,17.381056,42.545583,4.0,0.0,0.0,0.0,0.067114,0.932886,0.0,POINT (42.54558 17.38106)
956,ECE42DD0-2662-4317-8D57-A84C59218B43,1606,ABH-STD-BRIDEGMATTAR-SR-,ABHA,18.253889,42.628056,4.0,0.0,0.0,0.0,0.120743,0.879257,0.0,POINT (42.62806 18.25389)


In [204]:
import time
import tqdm

In [210]:
address_list = []
for item in tqdm.tqdm(branch_gdf.geometry.values):
    if (item.is_empty) or (item.x == 0) or (item.y ==0):
        address_list.append("")
    else:
        address = gpd.tools.reverse_geocode(item)
        address_list.append(address)
        time.sleep(1.5)

100%|██████████| 628/628 [23:44<00:00,  2.27s/it]


In [211]:
branch_gdf["address"] = address_list

In [213]:
branch_gdf.iloc[1]

BranchID                   88121164-31C6-4A58-BC4F-B424FBAE6DDC
Latitude                                                  26.38
Longitude                                                 43.91
cluster_id                                                    4
geometry          POINT (43.910000000000004 26.380000000000003)
address                           geometry  \
0  POINT (43.9...
Name: 1, dtype: object

In [222]:
address_list[2]["address"][0]

'Aroog Al-Shamal, طريق عمر بن الخطاب, 52379, طريق عمر بن الخطاب, بريدة, منطقة القصيم, السعودية'

In [278]:
colormap_clusters = {
    0: "green",
    1: "orange",
    2: "red",
    3: "blue",
    4: "pink",
    5: "gray",
    None: "white"
}

# 'red', 'blue', 'green', 'purple', 'orange', 'darkred',
#          'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue',
#          'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen',
#          'gray', 'black', 'lightgray'

In [279]:


coords = {
    "jeddah": [21.63, 39.17],
    "riyadh": [24.62, 46.71]
}

m = folium.Map(
    location=coords["jeddah"],
    zoom_start=10,
    # tiles="Cartodb Positron",
)
folium.GeoJson(
    branch_gdf,
    name=f"Petromin",
    zoom_on_click=True,
    marker=folium.Marker(icon=folium.Icon(icon='star')),
    tooltip=folium.GeoJsonTooltip(fields=["BranchID"]),
    popup=folium.GeoJsonPopup(fields=["BranchID"]),
    style_function=lambda x: {
        'markerColor': colormap_clusters[x['properties']['cluster_id']],
    },
).add_to(m)

In [280]:
m

In [116]:
branch_gdf.iloc[[30]]

,BranchID,Latitude,Longitude,cluster_id,geometry
30,82027A10-FD59-4514-B740-63CDD825640B,46.77,24.7,0,POINT (24.7 46.77)


In [281]:
branch_gdf[(branch_gdf["BranchID"].str.contains("4932E"))]

,BranchID,BranchCode,BranchName,City,Latitude,Longitude,cluster_id,cluster_0_trx_share,cluster_1_trx_share,cluster_2_trx_share,cluster_3_trx_share,cluster_4_trx_share,cluster_5_trx_share,geometry
25,E62E972E-0CB5-4E0E-9C5C-79CB83E4932E,2244,Al Asmary,ABHA,0.0,0.0,4.0,0.0,0.0,0.0,0.058824,0.941176,0.0,POINT (0 0)
969,E62E972E-0CB5-4E0E-9C5C-79CB83E4932E,2244,Al Asmary,ABHA,0.0,0.0,4.0,0.0,0.0,0.0,0.058824,0.941176,0.0,POINT (0 0)


In [124]:
branch_gdf[(branch_gdf["cluster_id"] == 4) & (branch_gdf["Latitude"] > 24)]

,BranchID,Latitude,Longitude,cluster_id,geometry
1,88121164-31C6-4A58-BC4F-B424FBAE6DDC,43.910000,26.380000,4,POINT (26.38 43.91)
2,C22A4104-43EA-4489-9E45-BE825185D4DA,43.920000,26.410000,4,POINT (26.41 43.92)
3,5426840E-07C4-4F13-A227-D30A9F057CEE,43.900000,26.370000,4,POINT (26.37 43.9)
4,B1323649-5458-41A4-B03D-C584CE74AF3E,43.920000,26.400000,4,POINT (26.4 43.92)
5,685C95DC-5ECC-4A91-BFAB-969557F0D48D,43.980000,26.340000,4,POINT (26.34 43.98)
...,...,...,...,...,...
620,AD7EE15B-2F65-4679-BB04-B916BFE54D0B,42.614167,20.003611,4,POINT (20.00361 42.61417)
621,87BE27E2-7CA6-48E4-8CF5-B92527E9607B,42.491389,18.207778,4,POINT (18.20778 42.49139)
622,62FCD4AA-53B1-4577-AFEF-9DB682E5E2C0,42.545583,17.381056,4,POINT (17.38106 42.54558)
623,ECE42DD0-2662-4317-8D57-A84C59218B43,42.628056,18.253889,4,POINT (18.25389 42.62806)


In [7]:
cities = df['City'].value_counts(100)
cities[cities > 0.01]

City
RIYADH       0.305031
JEDDAH       0.170860
DAMMAM       0.069182
MADINA       0.035639
Al Hassa     0.034591
MAKKAH       0.032495
Al Khobar    0.030398
KHAMIS       0.025157
TAIF         0.020964
JUBAIL       0.017820
QATIF        0.016771
YANBU        0.016771
JIZAN        0.015723
ABHA         0.015723
TABUK        0.014675
Buraydah     0.013627
HAIL         0.011530
AL KHARJ     0.010482
AHSA         0.010482
Name: proportion, dtype: float64

In [14]:
df[df["IsActive"] == True]["Longitude"].isna().mean()

0.06684856753069578

In [11]:
cities_na = df.dropna(subset="Longitude")['City'].value_counts(100)
cities_na[cities_na > 0.01]

City
RIYADH       0.308642
JEDDAH       0.161728
DAMMAM       0.069136
Al Hassa     0.039506
MADINA       0.035802
Al Khobar    0.035802
MAKKAH       0.033333
TAIF         0.024691
KHAMIS       0.024691
JUBAIL       0.018519
Buraydah     0.016049
YANBU        0.016049
TABUK        0.016049
QATIF        0.016049
JIZAN        0.014815
ABHA         0.013580
AL KHARJ     0.012346
Name: proportion, dtype: float64

In [13]:
cities_na = df[df["IsActive"] == True].dropna(subset="Longitude")['City'].value_counts(100)
cities_na[cities_na > 0.01]

City
RIYADH       0.309942
JEDDAH       0.146199
DAMMAM       0.073099
Al Hassa     0.045322
Al Khobar    0.042398
MADINA       0.036550
MAKKAH       0.033626
KHAMIS       0.026316
TAIF         0.026316
JUBAIL       0.019006
TABUK        0.017544
Buraydah     0.017544
JIZAN        0.016082
YANBU        0.014620
ABHA         0.013158
AL KHARJ     0.011696
HAIL         0.011696
QATIF        0.010234
Name: proportion, dtype: float64

In [ ]:
# TODO: function to get lat/long of address for each station
# TODO: feature - get another station at diff distances